# Generation of ORF consensus sequences for each unique barcode sequence

Before we generate the ORF consensus sequences, we will sort`orfs_bc_corrected_window4_dist4_ratio2.bam` by barcode sequence:

In [3]:
%%bash

DATA_DIR=/media/niek/4TB_SSD2/analyses/GPS_ONT
BAM_FILE=$DATA_DIR/orfs_bc_corrected_window4_dist4_ratio2.bam
SORTED_BAM_FILE=$DATA_DIR/orfs_bc_sorted.bam

samtools sort -@ 6 -t BC -o $SORTED_BAM_FILE $BAM_FILE

[bam_sort_core] merging from 3 files and 6 in-memory blocks...


## Creating consensus sequences

Strategy to create consensus sequences for each unique barcode sequence:

* Linear Streaming with groupby: Since the BAM file is pre-sorted by the BC (barcode) tag, the script uses itertools.groupby to process reads in "chunks." This avoids loading the entire multi-gigabyte BAM into RAM, storing only the reads for a single barcode at any given time.

* Reference-Guided Indexing: utilise pysam.FastaFile to access the GenCode cDNA library. This provides "random access," allowing the script to instantly fetch the specific "original" cDNA sequence for a given transcript ID without reading the entire FASTA file into memory.

* Majority-Reference Assignment: Because a single barcode might occasionally contain reads mapping to different isoforms (due to mapping ambiguity or chimeric reads), the script performs a "majority vote" to identify the primary cDNA target for each barcode.

* Positional Pileup (The Consensus Engine):

    - It initializes a "pileup table" exactly the length of the target cDNA.

    - It uses get_aligned_pairs() to map every base of every read to its specific coordinate on the cDNA reference, accounting for insertions and deletions (indels) defined in the CIGAR string.

* Base Calling: At every position in the cDNA, the script counts the frequency of A, C, G, T, and Gaps (-). The most frequent character is selected for the consensus.

* Gap Handling (Sequence Compression): If the majority "base" at a position is a deletion (meaning most reads agree a base is missing compared to the reference), it is skipped. This ensures the final consensus cDNA sequence is a continuous, biological sequence ready for translation or analysis.

* Data Integration: Finally, it writes the barcode, the raw GenCode metadata, the original reference sequence, and the newly generated consensus into a row-based format for export to CSV.

In [19]:
%%bash

DATA_DIR=/media/niek/4TB_SSD2/analyses/GPS_ONT
TEST_BAM_FILE=$DATA_DIR/orfs_bc_sorted_test.bam

python 07_orf_consensus.py \
	--input $TEST_BAM_FILE \
	--output $DATA_DIR/orfs_test_consensus.csv \
	--ref-fasta gencode.v49.pc_transcripts.fa


Loading reference FASTA: gencode.v49.pc_transcripts.fa
Processing barcodes and generating consensus...


1it [00:00,  1.58it/s]


Aggregating 1 results into DataFrame...
Done! Results saved to /media/niek/4TB_SSD2/analyses/GPS_ONT/orfs_test_consensus.csv
